In [1]:
from pathlib import Path
import pandas as pd
from scipy.stats import chi2_contingency
import plotly.express as px

pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)

In [2]:
oac_df = pd.read_csv('data/2021OAC/uk_oac_final.csv')
oac_df.rename(columns={'GeographyCode': 'OA'}, inplace=True)
oac_df.set_index('OA', inplace=True)
oac_df['Supergroup'] = oac_df['Supergroup'].astype('str')
oac_df.head()

,LAD25Code,LAD25Name,Supergroup,Group,Subgroup
OA,,,,,
E00000001,E09000001,City of London,3,3c,3c2
E00000003,E09000001,City of London,3,3c,3c2
E00000005,E09000001,City of London,3,3c,3c1
E00000007,E09000001,City of London,3,3c,3c1
E00000010,E09000001,City of London,3,3b,3b1


In [3]:
aemose_df = pd.read_parquet('scratch/ukc2021-proc20251009001rs__aemose-4-4_e7x3_s192-128-96-64_sage128_g0-64-32-g1-32-16_b512_20260517124211/ukc2021-proc20251009001rs__aemose-4-4_e7x3_s192-128-96-64_sage128_g0-64-32-g1-32-16_b512_20260517124211__ep1381-st516868__latent.parquet')
aemose_df["top_group_index"] = aemose_df["top_group_index"].astype(str)
aemose_df.head()

,index,train_valid_test,EMB_000,EMB_001,EMB_002,EMB_003,EMB_004,EMB_005,EMB_006,EMB_007,EMB_008,EMB_009,EMB_010,EMB_011,EMB_012,EMB_013,EMB_014,EMB_015,EMB_016,EMB_017,EMB_018,EMB_019,EMB_020,EMB_021,EMB_022,EMB_023,EMB_024,EMB_025,EMB_026,EMB_027,EMB_028,EMB_029,EMB_030,EMB_031,EMB_032,EMB_033,EMB_034,EMB_035,EMB_036,EMB_037,EMB_038,EMB_039,EMB_040,EMB_041,EMB_042,EMB_043,EMB_044,EMB_045,EMB_046,EMB_047,EMB_048,EMB_049,EMB_050,EMB_051,EMB_052,EMB_053,EMB_054,EMB_055,EMB_056,EMB_057,EMB_058,EMB_059,EMB_060,EMB_061,EMB_062,EMB_063,EXR_000,EXR_001,EXR_002,EXR_003,EXR_004,EXR_005,EXR_006,EXR_007,EXR_008,EXR_009,EXR_010,EXR_011,EXR_012,EXR_013,EXR_014,EXR_015,EXR_016,EXR_017,EXR_018,EXR_019,EXR_020,top_expert_index,top_group_index,EXR_entropy,top_expert_name
OA,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
E00071085,0,train,10.088778,-1.554822,1.773867,-3.237351,3.665862,-0.389270,5.771652,-0.436632,-12.981589,-0.628472,-5.419979,-1.865045,-3.269041,8.647029,7.049928,-1.654235,2.561972,1.255657,-11.170086,-1.623843,1.966225,-10.970493,0.751600,8.647429,10.130722,-1.379228,2.634787,-4.315769,-3.327369,1.412271,-2.259275,11.889605,0.391060,8.651897,-0.268966,0.438568,-2.202876,4.379434,9.826599,14.475407,-1.026285,5.697979,-0.868976,-5.116693,13.740377,-5.931295,0.023917,-5.830851,-9.739168,-6.682324,-5.604240,-1.393570,-10.639149,-2.977565,-4.925622,-6.133627,12.885193,-2.031326,-2.367699,1.344938,-5.750841,-9.652271,-17.059050,4.214541,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000e+00,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,10,3,-0.000000e+00,3_10
E00177567,1,valid,9.779602,-4.120526,3.127157,-1.140632,4.270104,0.594318,6.138867,1.075725,-12.327974,-2.621850,-6.726378,-2.316192,-2.881361,3.777979,6.764656,-0.982870,2.628485,3.489441,-12.032580,-1.664255,1.125723,-10.092644,-1.222035,7.362352,16.480036,-4.900255,1.327986,-2.187691,-3.765122,1.960589,-0.804136,9.322613,-0.095885,8.579175,-1.029155,0.957917,-1.541907,4.020084,9.724921,10.096922,-3.399083,6.228074,-0.397724,-3.201164,12.524422,-0.663969,-0.836373,-6.080329,-11.118063,-6.928390,-5.785718,-0.947352,-12.311965,-4.165529,-6.603142,-5.034491,15.211015,-2.087036,-5.251964,1.579664,-5.949459,-13.703210,-15.752306,3.387438,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,1.912222e-15,1.0,6.207677e-22,0.000000e+00,0.000000e+00,0.0,16,5,5.283300e-14,5_16
E00039966,2,test,10.403625,-1.787578,1.591250,-0.229739,5.158608,1.238132,5.078167,1.336170,-12.249456,-2.426197,-6.359148,-2.534961,-3.218826,10.011715,6.529750,-1.870262,2.692271,2.471268,-10.234532,-1.973777,1.321212,-11.120245,0.637222,6.734582,9.442896,-2.457126,4.427143,-3.493291,-3.923507,0.033155,-0.887652,10.661084,-0.716384,8.201168,-1.382604,0.124763,-2.423789,5.070688,8.953516,12.372397,-2.682573,4.819949,-2.007421,-4.457038,12.357480,-3.352425,1.830521,-7.091581,-10.825551,-5.860340,-3.887862,-0.355212,-10.776988,-2.850009,-3.915088,-7.226249,15.026868,-1.462750,-3.926569,1.445843,-6.046395,-7.380721,-15.996258,4.066959,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.416478e-10,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,9,3,9.512320e-09,3_09
E00155752,3,train,8.337117,-2.521118,3.724571,-1.342207,4.092932,-1.148114,6.089904,-1.506464,-14.459101,-0.628764,-4.595232,-2.985691,-1.003181,3.934668,6.481328,-0.873948,2.196750,0.576838,-10.148907,-2.902857,1.405432,-10.520889,-0.593642,6.251667,15.242721,-3.089197,1.971467,-5.455832,-5.739718,0.624366,-2.899794,11.657497,1.513073,8.553742,-1.206051,0.255119,-0.264144,6.192914,9.435658,13.565598,-1.921999,4.815767,-0.656067,-5.804866,11.815202,1.877431,-1.487086,-7.109060,-11.817463,-6.838669,-4.074008,-0.900709,-9.877106,-3.812966,-6.791752,-5.565237,17.579281,-2.173002,-3.481239,0.945997,-6.627783,-10.578916,-14.520760,3.819246,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.000000e+00,1.787199e-39,2.79

In [4]:
grp_oacsup_df  = aemose_df.join(oac_df, how="inner")
grp_oacsup_tab = pd.crosstab(grp_oacsup_df["Supergroup"], grp_oacsup_df["top_group_index"])
display(grp_oacsup_tab)

top_group_index,0,1,2,3,4,5,6
Supergroup,,,,,,,
1,12573,543,5824,7642,7936,0,7614
2,8912,3346,3889,10443,11095,0,11834
3,149,116,3532,64,447,15746,796
4,496,2741,4421,746,1565,11440,911
5,8828,558,1296,6181,2214,411,2375
6,1947,10975,2833,4146,4240,2016,9212
7,475,14563,6504,1335,6350,0,2453
8,493,1973,7276,25,1838,62,3623


In [5]:
grp_oacsup_stat, grp_oacsup_p, grp_oacsup_dof, _ = chi2_contingency(grp_oacsup_tab)
print(f"Chi-Square Statistic: {grp_oacsup_stat:.4f}")
print(f"P-value:              {grp_oacsup_p:.4f}")
print(f"Degrees of Freedom:   {grp_oacsup_dof}")

Chi-Square Statistic: 233307.1490
P-value:              0.0000
Degrees of Freedom:   42


In [6]:
exp_oacgrp_df  = aemose_df.join(oac_df, how="inner")
exp_oacgrp_tab = pd.crosstab(exp_oacgrp_df["Group"], exp_oacgrp_df["top_expert_name"])
display(exp_oacgrp_tab)

top_expert_name,0_00,0_01,0_02,1_03,1_04,1_05,2_06,2_07,2_08,3_09,3_10,3_11,4_12,4_13,4_14,5_15,5_16,5_17,6_18,6_19,6_20
Group,,,,,,,,,,,,,,,,,,,,,
1a,809,2491,46,0,30,4,936,0,27,114,575,0,42,948,3880,0,0,0,586,5376,3
1b,1234,2481,276,0,355,154,2035,1,130,714,4937,106,234,837,587,0,0,0,297,43,15
1c,1304,3431,501,0,0,0,2691,4,0,781,384,31,91,738,579,0,0,0,432,848,14
2a,1399,646,274,47,984,907,1061,13,321,3419,2034,322,388,1221,576,0,0,0,918,234,109
2b,1984,1080,157,11,435,298,806,2,210,1603,1364,80,277,1869,3821,0,0,0,575,3921,19
2c,1876,1060,436,10,361,293,954,8,514,873,670,78,420,1787,736,0,0,0,4299,1109,650
3a,2,1,18,5,1,3,64,2103,109,4,0,7,167,12,2,3289,7,25,67,0,361
3b,1,0,9,87,2,1,0,801,11,0,0,3,174,1,0,1558,2580,1558,1,0,103
3c,0,0,118,17,0,0,1,440,3,1,0,49,91,0,0,5379,503,847,1,0,263


In [7]:
exp_oacgrp_stat, exp_oacgrp_p, exp_oacgrp_dof, _ = chi2_contingency(exp_oacgrp_tab)
print(f"Chi-Square Statistic: {exp_oacgrp_stat:.4f}")
print(f"P-value:              {exp_oacgrp_p:.4f}")
print(f"Degrees of Freedom:   {exp_oacgrp_dof}")

Chi-Square Statistic: 710773.8753
P-value:              0.0000
Degrees of Freedom:   400
